In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
from delta.tables import DeltaTable

In [0]:
# =========================================================
# CONFIGURAÇÃO DAS TABELAS
# =========================================================
tables_config = [
    {
        "name": "denderecos",
        "staging_table": "assaia_catalog.google_drive.enderecos_clientes_enderecos_clientes_csv",
        "target_table": "assaia_catalog.bronze.denderecos",
        "pk_cols": ["_line"], 
        "watermark_col": "_fivetran_synced"
        }
    ,
    {
        "name": "dclientes",
        "staging_table": "assaia_catalog.google_drive.clientes_clientes_csv",
        "target_table": "assaia_catalog.bronze.dclientes",
        "pk_cols": ["_line"],
        "watermark_col": "_fivetran_synced"
        }
    ,
    {
        "name": "dprodutos",
        "staging_table": "assaia_catalog.google_drive.produtos_produtos_csv",
        "target_table": "assaia_catalog.bronze.dprodutos",
        "pk_cols": ["_line"],
        "watermark_col": "_fivetran_synced"
        }
    ,
    {
        "name": "dgerentes",
        "staging_table": "assaia_catalog.google_drive.gerentes_gerentes_csv",
        "target_table": "assaia_catalog.bronze.dgerentes",
        "pk_cols": ["_line"],
        "watermark_col": "_fivetran_synced"
        }
    ,
    {
        "name": "dlojas",
        "staging_table": "assaia_catalog.google_drive.lojas_lojas_csv",
        "target_table": "assaia_catalog.bronze.dlojas",
        "pk_cols": ["_line"],
        "watermark_col": "_fivetran_synced"
        }
    ,
    {
        "name": "fvendas", 
        "staging_table": "assaia_catalog.google_drive.vendas_vendas_csv",
        "target_table": "assaia_catalog.bronze.fvendas",
        "pk_cols": ["_line"],
        "watermark_col": "_fivetran_synced"
    }
]



In [0]:
# =========================================================
# FUNÇÃO DE INGESTÃO
# =========================================================


def ingest_table(cfg):
    name = cfg["name"]
    staging_table = cfg["staging_table"]
    target_table = cfg["target_table"]
    pk_cols = cfg["pk_cols"]
    watermark_col = cfg["watermark_col"]

    print(f"\n{'='*80}")
    print(f"Iniciando ingestão da tabela: {name}")
    print(f"Origem : {staging_table}")
    print(f"Destino: {target_table}")

    # ===== 1) Descobre watermark já processado =====
    if spark.catalog.tableExists(target_table):
        last_wm = (
            spark.table(target_table)
            .agg(F.max(F.col(watermark_col)).alias("last_wm"))
            .collect()[0]["last_wm"]
        )
    else:
        last_wm = None

    print(f"Último watermark encontrado: {last_wm}")

    # ===== 2) Lê dados da staging =====
    src = spark.table(staging_table)

    # ===== 3) Filtra incremental =====
    if last_wm is not None:
        src = src.filter(F.col(watermark_col) > F.lit(last_wm))

    # Se não houver novos dados, encerra
    if src.limit(1).count() == 0:
        print(f"Nenhum dado novo para processar em {name}.")
        return
    
    # ===== 5) Deduplicação =====
    win = Window.partitionBy(*pk_cols).orderBy(F.col(watermark_col).desc())

    src_dedup = (
##        src_mapped
        src
        .withColumn("_rn", F.row_number().over(win))
        .filter(F.col("_rn") == 1)
        .drop("_rn")
    )

    qtd = src_dedup.count()
    print(f"Quantidade de registros após deduplicação: {qtd}")

    # ===== 6) Se tabela destino não existir, cria =====
    if not spark.catalog.tableExists(target_table):
        print(f"Tabela destino não existe. Criando: {target_table}")
        (
            src_dedup.write
            .format("delta")
            .mode("overwrite")
            .saveAsTable(target_table)
        )
        print(f"Tabela {target_table} criada com sucesso.")
        return

    # ===== 7) MERGE (UPSERT) =====
    tgt = DeltaTable.forName(spark, target_table)

    join_cond = " AND ".join([f"t.{c} = s.{c}" for c in pk_cols])

    (
        tgt.alias("t")
        .merge(src_dedup.alias("s"), join_cond)
        .whenMatchedUpdateAll()
        .whenNotMatchedInsertAll()
        .execute()
    )

    print(f"Ingestão finalizada com sucesso para {name}.")

# =========================================================
# EXECUÇÃO PARA TODAS AS TABELAS
# =========================================================
for cfg in tables_config:
    try:
        ingest_table(cfg)
    except Exception as e:
        print(f"Erro ao processar {cfg['name']}: {str(e)}")